In [1]:
# Local environment check - dynamically load local_setup if running locally in VSCode
try:
    dbutils
except NameError:
    import os, sys
    from pathlib import Path
    
    # Walk up to locate project root folder
    curr_path = Path(os.getcwd()).resolve()
    project_root = None
    for _ in range(5):
        if (curr_path / "local_setup.py").exists():
            project_root = curr_path
            break
        curr_path = curr_path.parent
        
    if project_root:
        sys.path.append(str(project_root))
        from local_setup import spark, dbutils, display
    else:
        print("Warning: Could not locate local_setup.py in parent directories!")


Initializing local Spark session with Delta Lake support...
:: loading settings :: url = jar:file:/home/mi/.pyenv/versions/3.10.14/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/mi/.ivy2/cache
The jars for the packages stored in: /home/mi/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-19f00f5d-a698-4a80-a18a-2fd848328e2c;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.0 in central
	found io.delta#delta-storage;3.3.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 179ms :: artifacts dl 7ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.0 from central in [default]
	io.delta#delta-storage;3.3.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0 

Spark session successfully created.


In [2]:
import os
import glob
import sys

# Define widgets for dynamic ClientContainer selection
try:
    dbutils.widgets.text("ClientContainer", "new", "Client Container / Catalog Name")
    client_container = dbutils.widgets.get("ClientContainer").strip()
except Exception:
    client_container = "new"

def execute_ddl_scripts(ddl_directory: str, client_container: str):
    sql_files = sorted(glob.glob(os.path.join(ddl_directory, "*.sql")))
    
    if not sql_files:
        print(f"No .sql files found in: {ddl_directory}")
        return

    print(f"Processing {len(sql_files)} SQL file(s) from {ddl_directory}...")
    
    failed_files = []

    for sql_file in sql_files:
        file_name = os.path.basename(sql_file)
        
        try:
            with open(sql_file, 'r', encoding='utf-8') as f:
                sql_content = f.read()
            
            # Replace catalog placeholders with backticks for numerical safety (e.g. `274`)
            sql_content = sql_content.replace("new.silver.", f"`{client_container}`.silver.")
            sql_content = sql_content.replace("new.gold.", f"`{client_container}`.gold.")
            
            # Execute statement in Spark
            spark.sql(sql_content)
            print(f"SUCCESS: {file_name}")
            
        except Exception as e:
            print(f"FAILED: {file_name} | Error: {str(e)}")
            failed_files.append((file_name, str(e)))

    # Final summary check
    if failed_files:
        print(f"\nExecution finished with {len(failed_files)} failure(s):")
        for file, error in failed_files:
            print(f"  - {file}: {error[:100]}")
        raise RuntimeError("DDL batch execution failed.")
    
    print("All DDL scripts executed successfully.")

26/06/29 16:22:13 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [3]:
if __name__ == "__main__":
    # Resolve DDL directory dynamically relative to current notebook workspace location
    import os
    current_dir = os.getcwd()
    DDL_DIR = os.path.abspath(os.path.join(current_dir, "..", "DDL", "DimProvider"))
    execute_ddl_scripts(DDL_DIR, client_container)

Processing 2 SQL file(s) from /home/mi/Desktop/claim processing/claimprocessing_provider274/DDL/DimProvider...
[LOCAL SETUP] Table gold.dimprovider already exists. Skipping DDL execution.
SUCCESS: gold_dimprovider.sql
[LOCAL SETUP] Table silver.provider_hierarchy already exists. Skipping DDL execution.
SUCCESS: silver_provider_hierarchy.sql
All DDL scripts executed successfully.
